# Notebook 142: モデルベース強化学習の基礎 ― 「夢の中で学ぶ」

## Model-Based Reinforcement Learning Fundamentals: Learning in Dreams

---

### このノートブックの位置づけ

**世界モデル（World Models）シリーズ** の導入章として、モデルベース強化学習の基礎概念と Dyna-Q アルゴリズムを実装します。エージェントが「想像（imagination）」の中で学習する仕組みを、シンプルな GridWorld 環境で体感します。

### 学習目標

1. **モデルフリー vs モデルベース RL** の違いを理解する
2. **世界モデルの3要素**（表現モデル／遷移モデル／報酬モデル）を理解する
3. **Dyna-Q アルゴリズム** をゼロから実装する
4. **imagination 回数** が学習効率に与える影響を比較・分析する

### 前提知識

- MLP の基礎（Notebook 07）
- 勾配降下法の基本（Notebook 110）
- 世界モデルの概要（Notebook 140）

### 難易度: ★★★★☆ | 所要時間: 150〜180分

---

## 目次

1. [強化学習の基本概念](#1-強化学習の基本概念)
2. [モデルフリー vs モデルベース](#2-モデルフリー-vs-モデルベース)
3. [GridWorld 環境の実装](#3-gridworld-環境の実装)
4. [Q-Learning（モデルフリー）の実装](#4-q-learningモデルフリーの実装)
5. [世界モデルの3要素](#5-世界モデルの3要素)
6. [Dyna-Q の実装](#6-dyna-q-の実装)
7. [imagination 回数の影響比較](#7-imagination-回数の影響比較)
8. [方策の可視化](#8-方策の可視化)
9. [モデルベース RL の限界と展望](#9-モデルベース-rl-の限界と展望)
10. [まとめ・よくあるエラー・確認クイズ](#10-まとめよくあるエラー確認クイズ)

In [ ]:
# 環境セットアップ
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定（環境に応じて調整）
plt.rcParams['font.family'] = ['Hiragino Sans', 'Arial Unicode MS', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# 再現性のためのシード
np.random.seed(42)

print("環境セットアップ完了")
print(f"NumPy version: {np.__version__}")

---

## 1. 強化学習の基本概念

### 1.1 強化学習とは

強化学習（Reinforcement Learning, RL）は、**エージェント** が **環境** と相互作用しながら、**報酬** を最大化する行動方策を学習する枠組みです。

教師あり学習との最大の違いは、**正解ラベルが与えられない** ことです。エージェントは自分で行動を選び、その結果（報酬）から学びます。

### 1.2 RL ループの構造

```
┌──────────────┐
│   エージェント   │
│  (Agent)      │
└──┬───────▲───┘
   │行動(a) │状態(s'), 報酬(r)
   ▼       │
┌──────────────┐
│    環境       │
│ (Environment) │
└──────────────┘
```

各タイムステップで：
1. エージェントは現在の **状態 (State)** $s_t$ を観測する
2. **方策 (Policy)** $\pi$ に従って **行動 (Action)** $a_t$ を選択する
3. 環境は **次の状態** $s_{t+1}$ と **報酬** $r_t$ を返す
4. エージェントはこの経験 $(s_t, a_t, r_t, s_{t+1})$ から学習する

### 1.3 マルコフ決定過程 (MDP)

強化学習の問題は **マルコフ決定過程 (Markov Decision Process)** として定式化されます：

$$\text{MDP} = (\mathcal{S}, \mathcal{A}, P, R, \gamma)$$

| 要素 | 記号 | 意味 |
|------|------|------|
| 状態空間 | $\mathcal{S}$ | 取りうるすべての状態の集合 |
| 行動空間 | $\mathcal{A}$ | 取りうるすべての行動の集合 |
| 遷移関数 | $P(s'|s,a)$ | 状態 $s$ で行動 $a$ を取ったとき、状態 $s'$ に遷移する確率 |
| 報酬関数 | $R(s,a)$ | 状態 $s$ で行動 $a$ を取ったときに得られる報酬 |
| 割引率 | $\gamma \in [0,1]$ | 将来の報酬をどれだけ重視するか |

### 1.4 主要な用語

| 用語 | 英語 | 説明 |
|------|------|------|
| 方策 | Policy ($\pi$) | 状態から行動への写像。エージェントの「戦略」 |
| 価値関数 | Value Function ($V$) | ある状態からの期待累積報酬 |
| 行動価値関数 | Action-Value ($Q$) | ある状態で特定の行動を取ったときの期待累積報酬 |
| エピソード | Episode | 開始状態から終了状態までの一連の経験 |
| 探索 | Exploration | 未知の行動を試すこと |
| 活用 | Exploitation | 現在最良と思われる行動を選ぶこと |

---

## 2. モデルフリー vs モデルベース

### 2.1 二つのアプローチ

強化学習には大きく分けて **2つのアプローチ** があります。

#### モデルフリー RL（Model-Free RL）

- 環境のモデル（遷移関数・報酬関数）を **学習しない**
- 実際の経験だけから直接、方策や価値関数を学習する
- 例：Q-Learning, SARSA, PPO, SAC

#### モデルベース RL（Model-Based RL）

- 環境のモデル（遷移関数・報酬関数）を **学習する**
- 学習したモデルを使って「想像」の中で追加の経験を生成し、計画を立てる
- 例：Dyna-Q, MBPO, DreamerV3

### 2.2 直感的な理解：運転の学習

| 観点 | モデルフリー | モデルベース |
|------|------------|------------|
| 例え | 実際に車を運転して学ぶ | シミュレーターで練習してから運転する |
| 経験の使い方 | 一度使ったら捨てる | モデルに蓄積して何度も再利用 |
| サンプル効率 | 低い（大量の経験が必要） | 高い（少ない経験で学習可能） |
| 計算コスト | 低い（経験から直接更新） | 高い（モデル学習＋計画が必要） |
| モデル誤差 | なし | あり（モデルが不正確だと誤った学習をする） |
| 適用例 | Atari, ロボット制御 | ロボティクス, ゲーム AI |

### 2.3 なぜモデルベースが重要か

現実世界では **経験を得るコストが高い**（ロボットを壊す、時間がかかる等）。モデルベース RL は **少ない実経験** で効率的に学習できるため、実世界応用で特に重要です。

> **核心的なアイデア**: 環境のモデルを学び、そのモデルの中で「夢を見る」ように仮想的な経験を生成して学習する。これが「夢の中で学ぶ」の意味です。

In [ ]:
# モデルフリー vs モデルベースの概念図
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- モデルフリー ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('モデルフリー RL', fontsize=14, fontweight='bold', color='#2196F3')

# エージェント
agent_box = mpatches.FancyBboxPatch((1, 5), 3, 2, boxstyle='round,pad=0.3',
                                     facecolor='#BBDEFB', edgecolor='#1976D2', linewidth=2)
ax.add_patch(agent_box)
ax.text(2.5, 6, 'エージェント\n(Q-table)', ha='center', va='center', fontsize=10, fontweight='bold')

# 環境
env_box = mpatches.FancyBboxPatch((1, 1), 3, 2, boxstyle='round,pad=0.3',
                                   facecolor='#C8E6C9', edgecolor='#388E3C', linewidth=2)
ax.add_patch(env_box)
ax.text(2.5, 2, '環境\n(実世界)', ha='center', va='center', fontsize=10, fontweight='bold')

# 矢印
ax.annotate('', xy=(1.5, 3.2), xytext=(1.5, 4.8),
            arrowprops=dict(arrowstyle='->', color='#F44336', lw=2))
ax.text(0.3, 4, '行動 a', fontsize=9, color='#F44336')

ax.annotate('', xy=(3.5, 4.8), xytext=(3.5, 3.2),
            arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=2))
ax.text(4.0, 4, "s', r", fontsize=9, color='#4CAF50')

# 直接学習の矢印
ax.annotate('', xy=(5.5, 6), xytext=(4.2, 6),
            arrowprops=dict(arrowstyle='->', color='#FF9800', lw=2))
ax.text(5.7, 6.3, 'Q値を\n直接更新', fontsize=9, color='#FF9800')

# --- モデルベース ---
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('モデルベース RL', fontsize=14, fontweight='bold', color='#E91E63')

# エージェント
agent_box2 = mpatches.FancyBboxPatch((1, 5), 3, 2, boxstyle='round,pad=0.3',
                                      facecolor='#F8BBD0', edgecolor='#C2185B', linewidth=2)
ax.add_patch(agent_box2)
ax.text(2.5, 6, 'エージェント\n(Q-table)', ha='center', va='center', fontsize=10, fontweight='bold')

# 環境
env_box2 = mpatches.FancyBboxPatch((1, 1), 3, 2, boxstyle='round,pad=0.3',
                                    facecolor='#C8E6C9', edgecolor='#388E3C', linewidth=2)
ax.add_patch(env_box2)
ax.text(2.5, 2, '環境\n(実世界)', ha='center', va='center', fontsize=10, fontweight='bold')

# 世界モデル
model_box = mpatches.FancyBboxPatch((5.5, 3), 3.5, 2, boxstyle='round,pad=0.3',
                                     facecolor='#FFF9C4', edgecolor='#F9A825', linewidth=2)
ax.add_patch(model_box)
ax.text(7.25, 4, '世界モデル\n(想像の環境)', ha='center', va='center', fontsize=10, fontweight='bold')

# 実環境との相互作用
ax.annotate('', xy=(1.5, 3.2), xytext=(1.5, 4.8),
            arrowprops=dict(arrowstyle='->', color='#F44336', lw=2))
ax.text(0.3, 4, '行動 a', fontsize=9, color='#F44336')

ax.annotate('', xy=(3.5, 4.8), xytext=(3.5, 3.2),
            arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=2))
ax.text(4.0, 4, "s', r", fontsize=9, color='#4CAF50')

# モデルへの学習
ax.annotate('', xy=(5.5, 3.5), xytext=(4.2, 2.5),
            arrowprops=dict(arrowstyle='->', color='#9C27B0', lw=2, linestyle='dashed'))
ax.text(4.5, 2.5, 'モデル\n学習', fontsize=8, color='#9C27B0')

# 想像の経験
ax.annotate('', xy=(5.5, 5.5), xytext=(4.2, 6),
            arrowprops=dict(arrowstyle='<-', color='#FF9800', lw=2, linestyle='dashed'))
ax.text(4.5, 6.3, '想像の経験で\nQ値を更新', fontsize=8, color='#FF9800')

plt.tight_layout()
plt.show()

print("【核心的な違い】")
print("モデルフリー: 実経験 → Q値更新（1回きり）")
print("モデルベース: 実経験 → モデル学習 → 想像の経験を何度も生成 → Q値更新（N回追加）")

---

## 3. GridWorld 環境の実装

### 3.1 環境の設計

5x5 の格子世界（GridWorld）を実装します。これは強化学習の基本概念を学ぶための最もシンプルな環境の一つです。

```
┌───┬───┬───┬───┬───┐
│ S │   │   │   │   │  S: スタート (0,0)
├───┼───┼───┼───┼───┤
│   │ W │   │   │   │  W: 壁（通過不可）
├───┼───┼───┼───┼───┤
│   │ W │   │ W │   │  G: ゴール (4,4)
├───┼───┼───┼───┼───┤
│   │   │   │ W │   │
├───┼───┼───┼───┼───┤
│   │   │   │   │ G │
└───┴───┴───┴───┴───┘
```

- **行動**: 上(0), 下(1), 左(2), 右(3)
- **報酬**: ステップごとに -1、ゴール到達で +10、壁に衝突で -5
- **終了条件**: ゴール到達 or 最大100ステップ

In [ ]:
class GridWorld:
    """
    5x5 GridWorld 環境
    
    状態: (row, col) のタプル
    行動: 0=上, 1=下, 2=左, 3=右
    報酬: -1（各ステップ）, +10（ゴール）, -5（壁に衝突）
    """
    
    def __init__(self, grid_size=5):
        self.grid_size = grid_size
        self.start = (0, 0)          # スタート位置
        self.goal = (4, 4)           # ゴール位置
        
        # 壁の位置（通過不可能なセル）
        self.walls = {(1, 1), (2, 1), (2, 3), (3, 3)}
        
        # 行動の定義: 上, 下, 左, 右
        self.action_names = ['上', '下', '左', '右']
        self.action_deltas = {
            0: (-1, 0),  # 上
            1: (1, 0),   # 下
            2: (0, -1),  # 左
            3: (0, 1),   # 右
        }
        self.n_actions = 4
        self.n_states = grid_size * grid_size
        
        # 状態のリセット
        self.state = None
        self.steps = 0
        self.max_steps = 100  # 最大ステップ数
    
    def reset(self):
        """環境をリセットし、初期状態を返す"""
        self.state = self.start
        self.steps = 0
        return self.state
    
    def step(self, action):
        """
        行動を実行し、(次の状態, 報酬, 終了フラグ) を返す
        
        Args:
            action: 0=上, 1=下, 2=左, 3=右
        
        Returns:
            next_state: (row, col) タプル
            reward: float
            done: bool
        """
        self.steps += 1
        
        # 行動による移動先を計算
        dr, dc = self.action_deltas[action]
        new_row = self.state[0] + dr
        new_col = self.state[1] + dc
        new_state = (new_row, new_col)
        
        # 壁との衝突判定
        if (new_row < 0 or new_row >= self.grid_size or
            new_col < 0 or new_col >= self.grid_size or
            new_state in self.walls):
            # 壁に衝突：位置は変わらず、ペナルティ
            reward = -5.0
            new_state = self.state
        elif new_state == self.goal:
            # ゴール到達
            reward = 10.0
        else:
            # 通常の移動
            reward = -1.0
        
        # 終了判定
        done = (new_state == self.goal) or (self.steps >= self.max_steps)
        
        self.state = new_state
        return new_state, reward, done
    
    def get_valid_states(self):
        """壁を除く有効な状態のリストを返す"""
        states = []
        for r in range(self.grid_size):
            for c in range(self.grid_size):
                if (r, c) not in self.walls:
                    states.append((r, c))
        return states
    
    def state_to_index(self, state):
        """(row, col) → 整数インデックス"""
        return state[0] * self.grid_size + state[1]
    
    def index_to_state(self, index):
        """整数インデックス → (row, col)"""
        return (index // self.grid_size, index % self.grid_size)


# 環境のテスト
env = GridWorld()
state = env.reset()
print(f"初期状態: {state}")
print(f"ゴール: {env.goal}")
print(f"壁: {env.walls}")
print(f"行動空間: {env.action_names}")

# テスト実行
print("\n--- テスト実行 ---")
state = env.reset()
test_actions = [1, 3, 3, 1, 1]  # 下, 右, 右, 下, 下
for a in test_actions:
    next_state, reward, done = env.step(a)
    print(f"  行動: {env.action_names[a]:>2} → 状態: {next_state}, 報酬: {reward:+.1f}, 終了: {done}")
    if done:
        break

In [ ]:
def render_grid(env, agent_pos=None, title='GridWorld 環境', ax=None):
    """
    GridWorld を可視化する
    
    Args:
        env: GridWorld インスタンス
        agent_pos: エージェントの現在位置 (row, col) or None
        title: プロットのタイトル
        ax: matplotlib axes（None の場合は新規作成）
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    
    n = env.grid_size
    
    # グリッドの描画
    for r in range(n):
        for c in range(n):
            # セルの色を決定
            if (r, c) in env.walls:
                color = '#455A64'   # 壁: ダークグレー
            elif (r, c) == env.goal:
                color = '#4CAF50'   # ゴール: 緑
            elif (r, c) == env.start:
                color = '#2196F3'   # スタート: 青
            else:
                color = '#FAFAFA'   # 通常: 白
            
            rect = plt.Rectangle((c, n - 1 - r), 1, 1,
                                  facecolor=color, edgecolor='#BDBDBD', linewidth=1.5)
            ax.add_patch(rect)
            
            # ラベル
            if (r, c) in env.walls:
                ax.text(c + 0.5, n - 1 - r + 0.5, 'W',
                        ha='center', va='center', fontsize=14, color='white', fontweight='bold')
            elif (r, c) == env.goal:
                ax.text(c + 0.5, n - 1 - r + 0.5, 'G',
                        ha='center', va='center', fontsize=14, color='white', fontweight='bold')
            elif (r, c) == env.start:
                ax.text(c + 0.5, n - 1 - r + 0.5, 'S',
                        ha='center', va='center', fontsize=14, color='white', fontweight='bold')
    
    # エージェントの描画
    if agent_pos is not None:
        circle = plt.Circle((agent_pos[1] + 0.5, n - 1 - agent_pos[0] + 0.5),
                            0.3, facecolor='#FF5722', edgecolor='white', linewidth=2)
        ax.add_patch(circle)
        ax.text(agent_pos[1] + 0.5, n - 1 - agent_pos[0] + 0.5, 'A',
                ha='center', va='center', fontsize=12, color='white', fontweight='bold')
    
    ax.set_xlim(0, n)
    ax.set_ylim(0, n)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(range(n))
    ax.set_yticklabels(range(n - 1, -1, -1))
    ax.set_xlabel('列 (col)', fontsize=11)
    ax.set_ylabel('行 (row)', fontsize=11)
    
    return ax


# GridWorld の可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左: 初期状態
render_grid(env, agent_pos=env.start, title='GridWorld: 初期状態', ax=axes[0])

# 右: 途中の状態
render_grid(env, agent_pos=(2, 2), title='GridWorld: 探索中', ax=axes[1])

# 凡例
legend_elements = [
    mpatches.Patch(facecolor='#2196F3', label='S: スタート (0,0)'),
    mpatches.Patch(facecolor='#4CAF50', label='G: ゴール (4,4)'),
    mpatches.Patch(facecolor='#455A64', label='W: 壁'),
    mpatches.Patch(facecolor='#FF5722', label='A: エージェント'),
    mpatches.Patch(facecolor='#FAFAFA', edgecolor='gray', label='通常セル'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5, fontsize=10,
           bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.show()

print("【GridWorld の報酬設計】")
print("  通常の移動:  -1（最短経路を促進）")
print("  壁に衝突:    -5（壁を避けることを学習）")
print("  ゴール到達:  +10（目標達成のインセンティブ）")

---

## 4. Q-Learning（モデルフリー）の実装

### 4.1 Q-Learning アルゴリズム

Q-Learning は最も基本的な **モデルフリー** の強化学習アルゴリズムです。

**Q値の更新則:**

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

- $\alpha$: 学習率（新しい情報をどれだけ取り入れるか）
- $\gamma$: 割引率（将来の報酬をどれだけ重視するか）
- $\epsilon$-greedy: 確率 $\epsilon$ でランダム行動（探索）、確率 $1-\epsilon$ で最良行動（活用）

In [ ]:
class QLearningAgent:
    """
    Q-Learning エージェント（モデルフリー）
    
    Q(s, a) テーブルを直接更新して方策を学習する。
    環境のモデル（遷移関数、報酬関数）は一切使わない。
    """
    
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.95, epsilon=0.1):
        """
        Args:
            n_states: 状態数
            n_actions: 行動数
            alpha: 学習率
            gamma: 割引率
            epsilon: 探索率（epsilon-greedy）
        """
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        
        # Q テーブルの初期化（すべて 0）
        self.q_table = np.zeros((n_states, n_actions))
    
    def choose_action(self, state_idx):
        """
        epsilon-greedy 方策で行動を選択
        
        Args:
            state_idx: 状態のインデックス
        
        Returns:
            action: 選択された行動
        """
        if np.random.random() < self.epsilon:
            # 探索: ランダムに行動を選択
            return np.random.randint(self.n_actions)
        else:
            # 活用: Q値が最大の行動を選択（タイブレークはランダム）
            q_values = self.q_table[state_idx]
            max_q = np.max(q_values)
            # 最大値を持つ行動が複数ある場合はランダムに選択
            max_actions = np.where(q_values == max_q)[0]
            return np.random.choice(max_actions)
    
    def update(self, state_idx, action, reward, next_state_idx, done):
        """
        Q値の更新（TD学習）
        
        Q(s,a) ← Q(s,a) + α[r + γ max_a' Q(s',a') - Q(s,a)]
        """
        current_q = self.q_table[state_idx, action]
        
        if done:
            # 終了状態では次の状態の価値は 0
            target = reward
        else:
            # TD ターゲット: r + γ max Q(s', a')
            target = reward + self.gamma * np.max(self.q_table[next_state_idx])
        
        # Q値の更新
        self.q_table[state_idx, action] += self.alpha * (target - current_q)


print("QLearningAgent クラスを定義しました")
print(f"  - epsilon-greedy 方策")
print(f"  - Q(s,a) テーブルによる行動価値の保存")
print(f"  - TD学習による更新")

In [ ]:
def train_agent(env, agent, n_episodes=500, verbose=True):
    """
    エージェントを環境で学習させる
    
    Args:
        env: GridWorld 環境
        agent: 学習エージェント
        n_episodes: エピソード数
        verbose: 学習進捗を表示するか
    
    Returns:
        episode_rewards: 各エピソードの累積報酬のリスト
        episode_steps: 各エピソードのステップ数のリスト
    """
    episode_rewards = []
    episode_steps = []
    
    for episode in range(n_episodes):
        state = env.reset()
        state_idx = env.state_to_index(state)
        total_reward = 0
        steps = 0
        
        while True:
            # 行動を選択
            action = agent.choose_action(state_idx)
            
            # 環境で行動を実行
            next_state, reward, done = env.step(action)
            next_state_idx = env.state_to_index(next_state)
            
            # Q値を更新
            agent.update(state_idx, action, reward, next_state_idx, done)
            
            total_reward += reward
            steps += 1
            state_idx = next_state_idx
            
            if done:
                break
        
        episode_rewards.append(total_reward)
        episode_steps.append(steps)
        
        # 進捗表示
        if verbose and (episode + 1) % 100 == 0:
            avg_reward = np.mean(episode_rewards[-50:])
            avg_steps = np.mean(episode_steps[-50:])
            print(f"  Episode {episode+1:>4d} | "
                  f"平均報酬(直近50): {avg_reward:>7.1f} | "
                  f"平均ステップ数: {avg_steps:>5.1f}")
    
    return episode_rewards, episode_steps


# Q-Learning エージェントの学習
print("=" * 60)
print("Q-Learning（モデルフリー）の学習開始")
print("=" * 60)

np.random.seed(42)
env = GridWorld()
q_agent = QLearningAgent(
    n_states=env.n_states,
    n_actions=env.n_actions,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1
)

q_rewards, q_steps = train_agent(env, q_agent, n_episodes=500)

print("\n学習完了!")

In [ ]:
# Q-Learning の学習結果を可視化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) エピソード報酬の推移
window = 20
rewards_smooth = np.convolve(q_rewards, np.ones(window)/window, mode='valid')
axes[0].plot(q_rewards, alpha=0.3, color='#2196F3', label='各エピソード')
axes[0].plot(range(window-1, len(q_rewards)), rewards_smooth,
             color='#1565C0', linewidth=2, label=f'移動平均({window})')
axes[0].set_xlabel('エピソード', fontsize=11)
axes[0].set_ylabel('累積報酬', fontsize=11)
axes[0].set_title('Q-Learning: エピソード報酬の推移', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# (2) Q値のヒートマップ（各状態の最大Q値）
n = env.grid_size
v_table = np.full((n, n), np.nan)
for r in range(n):
    for c in range(n):
        if (r, c) not in env.walls:
            idx = env.state_to_index((r, c))
            v_table[r, c] = np.max(q_agent.q_table[idx])

im = axes[1].imshow(v_table, cmap='YlOrRd', interpolation='nearest')
axes[1].set_title('Q-Learning: 状態価値 V(s) = max_a Q(s,a)', fontsize=13)
axes[1].set_xlabel('列 (col)', fontsize=11)
axes[1].set_ylabel('行 (row)', fontsize=11)
for r in range(n):
    for c in range(n):
        if (r, c) in env.walls:
            axes[1].text(c, r, 'W', ha='center', va='center', fontsize=12,
                        color='white', fontweight='bold',
                        bbox=dict(boxstyle='round', facecolor='#455A64'))
        elif not np.isnan(v_table[r, c]):
            axes[1].text(c, r, f'{v_table[r,c]:.1f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=axes[1], shrink=0.8)

# (3) ステップ数の推移
steps_smooth = np.convolve(q_steps, np.ones(window)/window, mode='valid')
axes[2].plot(q_steps, alpha=0.3, color='#4CAF50', label='各エピソード')
axes[2].plot(range(window-1, len(q_steps)), steps_smooth,
             color='#2E7D32', linewidth=2, label=f'移動平均({window})')
axes[2].set_xlabel('エピソード', fontsize=11)
axes[2].set_ylabel('ステップ数', fontsize=11)
axes[2].set_title('Q-Learning: エピソードあたりのステップ数', fontsize=13)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【Q-Learning の学習結果】")
print(f"  最終50エピソードの平均報酬: {np.mean(q_rewards[-50:]):.1f}")
print(f"  最終50エピソードの平均ステップ数: {np.mean(q_steps[-50:]):.1f}")

In [ ]:
def plot_policy_arrows(q_table, env, title='学習した方策', ax=None):
    """
    方策を矢印で可視化する（方策矢印ヒートマップ）
    
    各セルに最良行動の矢印と Q値に基づく背景色を表示する。
    
    Args:
        q_table: Q テーブル (n_states, n_actions)
        env: GridWorld 環境
        title: プロットのタイトル
        ax: matplotlib axes
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))
    
    n = env.grid_size
    
    # 行動に対応する矢印の方向（表示座標系: y軸が反転）
    arrow_dx = {0: 0, 1: 0, 2: -0.3, 3: 0.3}   # 上, 下, 左, 右
    arrow_dy = {0: 0.3, 1: -0.3, 2: 0, 3: 0}    # y軸反転に注意
    
    # 状態価値を計算（背景色用）
    v_values = np.full((n, n), np.nan)
    for r in range(n):
        for c in range(n):
            if (r, c) not in env.walls and (r, c) != env.goal:
                idx = env.state_to_index((r, c))
                v_values[r, c] = np.max(q_table[idx])
    
    # 価値に基づく背景色
    v_min = np.nanmin(v_values)
    v_max = np.nanmax(v_values)
    
    for r in range(n):
        for c in range(n):
            # セルの背景色
            if (r, c) in env.walls:
                color = '#455A64'
            elif (r, c) == env.goal:
                color = '#4CAF50'
            elif not np.isnan(v_values[r, c]):
                # 正規化した値を色にマッピング
                if v_max > v_min:
                    norm_v = (v_values[r, c] - v_min) / (v_max - v_min)
                else:
                    norm_v = 0.5
                color = plt.cm.YlOrRd(0.2 + 0.6 * norm_v)
            else:
                color = '#FAFAFA'
            
            rect = plt.Rectangle((c, n - 1 - r), 1, 1,
                                  facecolor=color, edgecolor='#BDBDBD', linewidth=1.5)
            ax.add_patch(rect)
            
            # 壁とゴールのラベル
            if (r, c) in env.walls:
                ax.text(c + 0.5, n - 1 - r + 0.5, 'W',
                        ha='center', va='center', fontsize=14, color='white', fontweight='bold')
            elif (r, c) == env.goal:
                ax.text(c + 0.5, n - 1 - r + 0.5, 'G',
                        ha='center', va='center', fontsize=16, color='white', fontweight='bold')
            else:
                # 最良行動の矢印を描画
                idx = env.state_to_index((r, c))
                best_action = np.argmax(q_table[idx])
                cx = c + 0.5
                cy = n - 1 - r + 0.5
                dx = arrow_dx[best_action]
                dy = arrow_dy[best_action]
                ax.annotate('', xy=(cx + dx, cy + dy), xytext=(cx, cy),
                            arrowprops=dict(arrowstyle='->', color='#1A237E',
                                           lw=2.5, mutation_scale=15))
    
    ax.set_xlim(0, n)
    ax.set_ylim(0, n)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(range(n))
    ax.set_yticklabels(range(n - 1, -1, -1))
    ax.set_xlabel('列 (col)', fontsize=11)
    ax.set_ylabel('行 (row)', fontsize=11)
    
    return ax


# Q-Learning で学習した方策を可視化
fig, ax = plt.subplots(figsize=(7, 7))
plot_policy_arrows(q_agent.q_table, env, title='Q-Learning で学習した方策', ax=ax)
plt.tight_layout()
plt.show()

print("【方策の読み方】")
print("  矢印: その状態で取るべき最良行動の方向")
print("  背景色: 状態価値（暖色ほど高い価値 = ゴールに近い）")
print("  W: 壁, G: ゴール")

---

## 5. 世界モデルの3要素

### 5.1 世界モデルとは

モデルベース RL における **世界モデル (World Model)** は、環境のダイナミクスを近似する内部モデルです。

世界モデルは以下の **3つの要素** から構成されます：

| 要素 | 役割 | 数式 | GridWorld での実装 |
|------|------|------|-------------------|
| **表現モデル** | 観測 → 潜在状態 | $z_t = f_{\text{enc}}(o_t)$ | 状態がすでに (row, col) なのでそのまま使用 |
| **遷移モデル** | 次の状態を予測 | $\hat{s}_{t+1} = f_{\text{trans}}(s_t, a_t)$ | 経験した遷移をテーブルに記録 |
| **報酬モデル** | 報酬を予測 | $\hat{r}_t = f_{\text{reward}}(s_t, a_t)$ | 経験した報酬をテーブルに記録 |

### 5.2 GridWorld での世界モデル

GridWorld は状態空間が離散的で小さいため、テーブルベースでモデルを構築できます：

- 実際に経験した $(s, a) \to (s', r)$ の遷移をすべて記録
- 「想像」時にはテーブルから $(s, a)$ をランダムに選び、記録された $(s', r)$ を返す

> **注意**: 実際の DreamerV3 などではニューラルネットワークで遷移モデルを学習しますが、概念的には同じことをしています。

In [ ]:
class EnvironmentModel:
    """
    環境モデル（テーブルベース）
    
    実際の経験 (s, a) → (s', r) を記録し、
    想像（imagination）時にはランダムに経験を再生する。
    
    世界モデルの3要素:
    - 表現モデル: GridWorld では状態がすでにコンパクト（省略）
    - 遷移モデル: (s, a) → s' の記録
    - 報酬モデル: (s, a) → r の記録
    """
    
    def __init__(self):
        # (state_idx, action) → (next_state_idx, reward) のマッピング
        self.transitions = {}  # {(s, a): (s', r)}
        # 経験した (state_idx, action) のペアを記録
        self.experienced_sa = []
    
    def store(self, state_idx, action, next_state_idx, reward):
        """
        実際の経験を環境モデルに記録する
        
        Args:
            state_idx: 現在の状態インデックス
            action: 実行した行動
            next_state_idx: 次の状態インデックス
            reward: 受け取った報酬
        """
        sa_pair = (state_idx, action)
        
        if sa_pair not in self.transitions:
            self.experienced_sa.append(sa_pair)
        
        # 最新の経験で上書き（決定的環境の場合）
        self.transitions[sa_pair] = (next_state_idx, reward)
    
    def simulate(self):
        """
        ランダムに過去の経験を選び、仮想的な経験を生成する
        （= 想像 / imagination）
        
        Returns:
            state_idx: 選ばれた状態
            action: 選ばれた行動
            next_state_idx: モデルが予測する次の状態
            reward: モデルが予測する報酬
        """
        # ランダムに過去の (s, a) ペアを選択
        idx = np.random.randint(len(self.experienced_sa))
        state_idx, action = self.experienced_sa[idx]
        next_state_idx, reward = self.transitions[(state_idx, action)]
        
        return state_idx, action, next_state_idx, reward
    
    def has_experience(self):
        """経験があるかどうかを返す"""
        return len(self.experienced_sa) > 0


# テスト
model = EnvironmentModel()

# いくつかの遷移を記録
model.store(0, 1, 5, -1.0)   # (0,0)から下に移動 → (1,0)
model.store(5, 3, 6, -1.0)   # (1,0)から右に移動 → (1,1)...壁の場合は別
model.store(0, 3, 1, -1.0)   # (0,0)から右に移動 → (0,1)

print("環境モデルのテスト:")
print(f"  記録された遷移数: {len(model.experienced_sa)}")
print(f"  記録内容: {model.transitions}")

# 想像の経験を生成
print("\n想像（imagination）の例:")
for i in range(5):
    s, a, ns, r = model.simulate()
    print(f"  想像 {i+1}: 状態{s}, 行動{a} → 次の状態{ns}, 報酬{r}")

---

## 6. Dyna-Q の実装

### 6.1 Dyna-Q アルゴリズム

Dyna-Q は **Q-Learning + 環境モデル + 想像（imagination）** を組み合わせたアルゴリズムです（Sutton, 1991）。

**各ステップでの処理:**

1. **実経験**: 環境で行動し、$(s, a, r, s')$ を得る
2. **直接 RL**: 実経験で Q 値を更新（通常の Q-Learning）
3. **モデル学習**: 経験を環境モデルに記録
4. **計画（Planning）**: 環境モデルから $N$ 回の想像経験を生成し、Q 値を追加更新

```
実世界で1回行動
    │
    ├─→ Q値を更新（通常のQ-Learning）
    ├─→ 環境モデルに (s,a,s',r) を記録
    └─→ N回の「想像」でQ値を追加更新  ← ここがDyna-Qの核心
         │
         ├─ 想像1回目: モデルから (s,a)→(s',r) を取得 → Q更新
         ├─ 想像2回目: モデルから (s,a)→(s',r) を取得 → Q更新
         ⋮
         └─ 想像N回目: モデルから (s,a)→(s',r) を取得 → Q更新
```

N が大きいほど、1回の実経験からより多くの学習ができます。

In [ ]:
class DynaQAgent:
    """
    Dyna-Q エージェント（モデルベース RL）
    
    Q-Learning に環境モデルと想像（imagination）を追加した
    モデルベース強化学習アルゴリズム。
    
    核心的なアイデア:
      実経験で学習 + モデルの中で「夢を見て」追加学習
    """
    
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.95,
                 epsilon=0.1, n_planning=5):
        """
        Args:
            n_states: 状態数
            n_actions: 行動数
            alpha: 学習率
            gamma: 割引率
            epsilon: 探索率
            n_planning: imagination の回数（N）
        """
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.n_planning = n_planning
        
        # Q テーブル
        self.q_table = np.zeros((n_states, n_actions))
        
        # 環境モデル
        self.model = EnvironmentModel()
    
    def choose_action(self, state_idx):
        """epsilon-greedy 方策で行動を選択"""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        else:
            q_values = self.q_table[state_idx]
            max_q = np.max(q_values)
            max_actions = np.where(q_values == max_q)[0]
            return np.random.choice(max_actions)
    
    def _update_q(self, state_idx, action, reward, next_state_idx, done):
        """Q値の更新（内部メソッド）"""
        current_q = self.q_table[state_idx, action]
        if done:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.q_table[next_state_idx])
        self.q_table[state_idx, action] += self.alpha * (target - current_q)
    
    def update(self, state_idx, action, reward, next_state_idx, done):
        """
        Dyna-Q の更新ステップ
        
        1. 実経験で Q 値を更新（直接 RL）
        2. 環境モデルに経験を記録（モデル学習）
        3. N 回の想像で Q 値を追加更新（計画）
        """
        # ステップ1: 直接 RL（通常の Q-Learning 更新）
        self._update_q(state_idx, action, reward, next_state_idx, done)
        
        # ステップ2: 環境モデルに経験を記録
        self.model.store(state_idx, action, next_state_idx, reward)
        
        # ステップ3: 想像（Planning）
        if self.model.has_experience():
            for _ in range(self.n_planning):
                # モデルからランダムに経験をサンプリング
                sim_s, sim_a, sim_ns, sim_r = self.model.simulate()
                # 想像の経験で Q 値を更新
                # 注意: 想像では done=False として扱う（簡略化）
                sim_done = (sim_ns == env.state_to_index(env.goal))
                self._update_q(sim_s, sim_a, sim_r, sim_ns, sim_done)


print("DynaQAgent クラスを定義しました")
print(f"  - Q-Learning + 環境モデル + imagination")
print(f"  - 各実ステップ後に N 回の想像で追加学習")
print(f"  - N=0 の場合は通常の Q-Learning と同等")

In [ ]:
# Dyna-Q (N=5) の学習
print("=" * 60)
print("Dyna-Q (N=5) の学習開始")
print("=" * 60)

np.random.seed(42)
env = GridWorld()
dyna_agent = DynaQAgent(
    n_states=env.n_states,
    n_actions=env.n_actions,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1,
    n_planning=5  # 各ステップで5回の想像
)

dyna_rewards, dyna_steps = train_agent(env, dyna_agent, n_episodes=500)

print("\n学習完了!")
print(f"  最終50エピソードの平均報酬: {np.mean(dyna_rewards[-50:]):.1f}")
print(f"  最終50エピソードの平均ステップ数: {np.mean(dyna_steps[-50:]):.1f}")

In [ ]:
# Q-Learning vs Dyna-Q (N=5) の比較
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

window = 20

# (1) 累積報酬の比較
q_smooth = np.convolve(q_rewards, np.ones(window)/window, mode='valid')
dyna_smooth = np.convolve(dyna_rewards, np.ones(window)/window, mode='valid')

axes[0].plot(range(window-1, len(q_rewards)), q_smooth,
             color='#2196F3', linewidth=2, label='Q-Learning（モデルフリー）')
axes[0].plot(range(window-1, len(dyna_rewards)), dyna_smooth,
             color='#E91E63', linewidth=2, label='Dyna-Q (N=5)（モデルベース）')
axes[0].set_xlabel('エピソード', fontsize=11)
axes[0].set_ylabel('累積報酬（移動平均）', fontsize=11)
axes[0].set_title('学習曲線の比較: 報酬', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# (2) ステップ数の比較
q_steps_smooth = np.convolve(q_steps, np.ones(window)/window, mode='valid')
dyna_steps_smooth = np.convolve(dyna_steps, np.ones(window)/window, mode='valid')

axes[1].plot(range(window-1, len(q_steps)), q_steps_smooth,
             color='#2196F3', linewidth=2, label='Q-Learning')
axes[1].plot(range(window-1, len(dyna_steps)), dyna_steps_smooth,
             color='#E91E63', linewidth=2, label='Dyna-Q (N=5)')
axes[1].set_xlabel('エピソード', fontsize=11)
axes[1].set_ylabel('ステップ数（移動平均）', fontsize=11)
axes[1].set_title('学習曲線の比較: ステップ数', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【比較結果】")
print(f"  Q-Learning:     平均報酬 = {np.mean(q_rewards[-50:]):>7.1f}, 平均ステップ = {np.mean(q_steps[-50:]):>5.1f}")
print(f"  Dyna-Q (N=5):   平均報酬 = {np.mean(dyna_rewards[-50:]):>7.1f}, 平均ステップ = {np.mean(dyna_steps[-50:]):>5.1f}")
print("\n→ Dyna-Q は想像による追加学習で、より早く良い方策を獲得!")

---

## 7. imagination 回数の影響比較

### 7.1 実験設定

Dyna-Q の想像回数 $N$ を変えて、学習効率への影響を調べます。

- $N = 0$: 想像なし（= 通常の Q-Learning）
- $N = 5$: 各ステップで 5 回の想像
- $N = 20$: 各ステップで 20 回の想像
- $N = 50$: 各ステップで 50 回の想像

仮説: **N が大きいほど学習が速い**（ただし計算コストも増加）

In [ ]:
# imagination 回数を変えた実験
n_planning_values = [0, 5, 20, 50]
results = {}  # {N: (rewards, steps, agent)}

for n_plan in n_planning_values:
    print(f"\n--- Dyna-Q (N={n_plan}) の学習 ---")
    
    np.random.seed(42)
    env = GridWorld()
    agent = DynaQAgent(
        n_states=env.n_states,
        n_actions=env.n_actions,
        alpha=0.1,
        gamma=0.95,
        epsilon=0.1,
        n_planning=n_plan
    )
    
    rewards, steps = train_agent(env, agent, n_episodes=500, verbose=False)
    results[n_plan] = (rewards, steps, agent)
    
    avg_r = np.mean(rewards[-50:])
    avg_s = np.mean(steps[-50:])
    print(f"  最終50ep: 平均報酬={avg_r:.1f}, 平均ステップ={avg_s:.1f}")

print("\nすべての実験が完了しました!")

In [ ]:
# 学習曲線の比較
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'0': '#2196F3', '5': '#4CAF50', '20': '#FF9800', '50': '#E91E63'}
window = 20

for n_plan in n_planning_values:
    rewards, steps, _ = results[n_plan]
    c = colors[str(n_plan)]
    label = f'N={n_plan}' if n_plan > 0 else 'N=0 (Q-Learning)'
    
    # 報酬の移動平均
    r_smooth = np.convolve(rewards, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(rewards)), r_smooth,
                 color=c, linewidth=2, label=label)
    
    # ステップ数の移動平均
    s_smooth = np.convolve(steps, np.ones(window)/window, mode='valid')
    axes[1].plot(range(window-1, len(steps)), s_smooth,
                 color=c, linewidth=2, label=label)

axes[0].set_xlabel('エピソード', fontsize=11)
axes[0].set_ylabel('累積報酬（移動平均）', fontsize=11)
axes[0].set_title('imagination 回数と学習速度：報酬', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('エピソード', fontsize=11)
axes[1].set_ylabel('ステップ数（移動平均）', fontsize=11)
axes[1].set_title('imagination 回数と学習速度：ステップ数', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【観察】")
print("1. N=0（Q-Learning）は最も学習が遅い")
print("2. N が増えるほど、序盤の学習が加速する")
print("3. 最終的にはすべての手法が同程度の性能に収束する")
print("4. 想像の回数を増やすことで、サンプル効率が大幅に改善される")

In [ ]:
# 累積報酬の比較（学習効率の定量的な比較）
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) 累積報酬
for n_plan in n_planning_values:
    rewards, _, _ = results[n_plan]
    cum_rewards = np.cumsum(rewards)
    c = colors[str(n_plan)]
    label = f'N={n_plan}' if n_plan > 0 else 'N=0 (Q-Learning)'
    axes[0].plot(cum_rewards, color=c, linewidth=2, label=label)

axes[0].set_xlabel('エピソード', fontsize=11)
axes[0].set_ylabel('累積報酬', fontsize=11)
axes[0].set_title('累積報酬の比較', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# (2) 「良い方策を獲得するまでの速度」
# 各エピソード区間での平均報酬
intervals = [(0, 50), (50, 100), (100, 200), (200, 500)]
interval_labels = ['0-50', '50-100', '100-200', '200-500']

x = np.arange(len(intervals))
width = 0.18

for i, n_plan in enumerate(n_planning_values):
    rewards, _, _ = results[n_plan]
    interval_means = [np.mean(rewards[s:e]) for s, e in intervals]
    c = colors[str(n_plan)]
    label = f'N={n_plan}' if n_plan > 0 else 'N=0'
    axes[1].bar(x + i * width, interval_means, width, color=c, label=label, alpha=0.8)

axes[1].set_xlabel('エピソード区間', fontsize=11)
axes[1].set_ylabel('平均報酬', fontsize=11)
axes[1].set_title('学習段階ごとの平均報酬', fontsize=13)
axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels(interval_labels)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("【定量的な比較（最終50エピソードの平均報酬）】")
for n_plan in n_planning_values:
    rewards, steps, _ = results[n_plan]
    r = np.mean(rewards[-50:])
    s = np.mean(steps[-50:])
    print(f"  N={n_plan:>2d}: 平均報酬 = {r:>7.1f}, 平均ステップ = {s:>5.1f}")

### 7.2 想像回数の収穫逓減とモデル誤差

imagination 回数を増やすと学習は加速しますが、**収穫逓減（diminishing returns）** が起きます。

**理由:**

1. **情報の飽和**: 同じ経験を何度も再生しても、新しい情報は得られない
2. **計算コスト**: N を2倍にしても、学習速度は2倍にはならない
3. **モデル誤差の蓄積**: 学習初期のモデルは不正確であり、不正確なモデルからの想像は誤った学習を引き起こす可能性がある

> **実用的な指針**: N は「大きければ大きいほど良い」わけではなく、環境の複雑さとモデルの精度に応じて適切に設定する必要があります。

---

## 8. 方策の可視化

### 8.1 各 imagination 回数での方策

学習した方策を矢印ヒートマップで可視化し、imagination 回数による違いを確認します。

In [ ]:
# 各 imagination 回数で学習した方策を比較
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

titles = {
    0: 'N=0 (Q-Learning: モデルフリー)',
    5: 'N=5 (Dyna-Q)',
    20: 'N=20 (Dyna-Q)',
    50: 'N=50 (Dyna-Q)'
}

for ax, n_plan in zip(axes.flatten(), n_planning_values):
    _, _, agent = results[n_plan]
    plot_policy_arrows(agent.q_table, env, title=titles[n_plan], ax=ax)

plt.suptitle('imagination 回数による方策の比較', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("【観察ポイント】")
print("1. すべてのバリエーションが壁を避けてゴールに向かう方策を学習")
print("2. N が大きいほど、方策がより洗練される（特に遠い状態で明確な方向を示す）")
print("3. N=0 でも最終的には良い方策を学習するが、途中経過の品質に差がある")

In [ ]:
# Q-Learning vs Dyna-Q (N=50) の方策を並べて比較
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Q-Learning
_, _, agent_n0 = results[0]
plot_policy_arrows(agent_n0.q_table, env,
                   title='Q-Learning (N=0): モデルフリー', ax=axes[0])

# Dyna-Q (N=50)
_, _, agent_n50 = results[50]
plot_policy_arrows(agent_n50.q_table, env,
                   title='Dyna-Q (N=50): モデルベース', ax=axes[1])

plt.suptitle('モデルフリー vs モデルベース: 方策の比較', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 価値関数ヒートマップの比較
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

n = env.grid_size

for ax, (n_plan, agent_label) in zip(axes, [(0, 'Q-Learning (N=0)'), (50, 'Dyna-Q (N=50)')]):
    _, _, agent = results[n_plan]
    v_table = np.full((n, n), np.nan)
    
    for r in range(n):
        for c in range(n):
            if (r, c) not in env.walls:
                idx = env.state_to_index((r, c))
                v_table[r, c] = np.max(agent.q_table[idx])
    
    im = ax.imshow(v_table, cmap='RdYlGn', interpolation='nearest',
                   vmin=np.nanmin(v_table), vmax=np.nanmax(v_table))
    
    for r in range(n):
        for c in range(n):
            if (r, c) in env.walls:
                ax.text(c, r, 'W', ha='center', va='center', fontsize=14,
                        color='white', fontweight='bold',
                        bbox=dict(boxstyle='round', facecolor='#455A64'))
            elif (r, c) == env.goal:
                ax.text(c, r, 'G', ha='center', va='center', fontsize=14,
                        color='white', fontweight='bold',
                        bbox=dict(boxstyle='round', facecolor='#4CAF50'))
            elif not np.isnan(v_table[r, c]):
                ax.text(c, r, f'{v_table[r,c]:.1f}', ha='center', va='center',
                        fontsize=10, fontweight='bold')
    
    ax.set_title(f'{agent_label}\n状態価値 V(s) = max_a Q(s,a)', fontsize=13)
    ax.set_xlabel('列 (col)', fontsize=11)
    ax.set_ylabel('行 (row)', fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.8, label='状態価値')

plt.suptitle('価値関数の比較', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("【価値関数の読み方】")
print("  緑色（高い値）: ゴールに近い、価値の高い状態")
print("  赤色（低い値）: ゴールから遠い、価値の低い状態")
print("  W: 壁（価値なし）、G: ゴール")

---

## 9. モデルベース RL の限界と展望

### 9.1 モデル誤差の蓄積問題

モデルベース RL の最大の課題は、**モデル誤差（Model Error）が長い時間軸で蓄積する** ことです。

In [ ]:
# モデル誤差の蓄積を可視化するシミュレーション
# 各ステップで小さな誤差が加わり、長いホライズンで大きなズレになることを示す

np.random.seed(42)

# 真の遷移（1次元の例）
true_position = 0.0
model_position = 0.0
step_size = 1.0
error_std = 0.1  # モデルの各ステップでの予測誤差

n_steps = 50
true_trajectory = [true_position]
model_trajectory = [model_position]
error_trajectory = [0.0]

for t in range(n_steps):
    # 真の遷移
    true_position += step_size
    true_trajectory.append(true_position)
    
    # モデルの予測（各ステップで小さな誤差が加わる）
    model_position += step_size + np.random.normal(0, error_std)
    model_trajectory.append(model_position)
    
    error_trajectory.append(abs(true_position - model_position))

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) 軌跡の比較
axes[0].plot(true_trajectory, 'b-', linewidth=2, label='真の軌跡')
axes[0].plot(model_trajectory, 'r--', linewidth=2, label='モデルの予測')
axes[0].fill_between(range(len(true_trajectory)),
                     [t - 2 for t in true_trajectory],
                     [t + 2 for t in true_trajectory],
                     alpha=0.1, color='blue')
axes[0].set_xlabel('タイムステップ', fontsize=11)
axes[0].set_ylabel('位置', fontsize=11)
axes[0].set_title('モデル予測の蓄積誤差', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# (2) 誤差の推移
axes[1].plot(error_trajectory, 'r-', linewidth=2)
axes[1].fill_between(range(len(error_trajectory)), 0, error_trajectory, alpha=0.3, color='red')

# 理論的な誤差成長（sqrt(t) に比例）
t_range = np.arange(1, n_steps + 1)
theory_error = error_std * np.sqrt(t_range)
axes[1].plot(t_range, theory_error, 'k--', linewidth=1.5,
             label=f'理論値: $\sigma\sqrt{{t}}$ ($\sigma$={error_std})')

axes[1].set_xlabel('タイムステップ', fontsize=11)
axes[1].set_ylabel('予測誤差', fontsize=11)
axes[1].set_title('時間に伴う誤差の成長', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【モデル誤差の蓄積】")
print("  各ステップの誤差が小さくても (σ=0.1)、")
print("  長いホライズンでは √t に比例して誤差が成長する。")
print(f"  t=50 での理論的誤差: {error_std * np.sqrt(50):.2f}")

### 9.2 モデルベース RL の主な課題

| 課題 | 説明 | 対策 |
|------|------|------|
| モデル誤差の蓄積 | 長期予測で誤差が増大 | 短いホライズンで計画、アンサンブルモデル |
| 高次元状態空間 | 画像など高次元の入力を扱えない | 表現学習（VAE, CNN）で潜在空間に圧縮 |
| 確率的環境 | 同じ (s,a) でも結果が異なる | 確率的モデル、分布予測 |
| 計算コスト | モデル学習 + 計画の計算量 | 効率的なアーキテクチャ設計 |

### 9.3 Notebook 140 との接続

Notebook 140 で概要を学んだ **世界モデル** の考え方は、まさにこの「モデルベース RL」の発展形です：

1. **Ha & Schmidhuber (2018)**: VAE + RNN で世界モデルを構築 → 「夢の中」で方策を学習
2. **DreamerV3 (2023)**: RSSM（Recurrent State Space Model）による高精度な世界モデル
3. **核心**: 表現学習 + 遷移モデル + 報酬モデル = 今回学んだ3要素と同じ構造

### 9.4 今後の学習への橋渡し

今回の GridWorld は状態空間が離散的で小さいため、テーブルベースの世界モデルで十分でした。しかし、現実世界の問題では：

- 状態が **画像**（高次元連続値）
- 遷移が **非線形・確率的**
- 時間的な **依存関係** が長い

これらを扱うために、**ニューラルネットワーク** による世界モデルが必要になります。次のノートブック以降で段階的に学んでいきます。

---

## 10. まとめ・よくあるエラー・確認クイズ

### 10.1 このノートブックで学んだこと

| 項目 | 内容 |
|------|------|
| **モデルフリー RL** | 環境のモデルを使わず、実経験だけから直接学習（Q-Learning） |
| **モデルベース RL** | 環境のモデルを学習し、想像の中で追加学習（Dyna-Q） |
| **世界モデルの3要素** | 表現モデル、遷移モデル、報酬モデル |
| **Dyna-Q** | Q-Learning + 環境モデル + imagination で学習効率を向上 |
| **imagination の効果** | N を増やすとサンプル効率が改善（ただし収穫逓減あり） |
| **モデル誤差** | モデルの不正確さが長期的に蓄積するリスク |

### 10.2 よくあるエラー 3選

#### エラー1: 環境モデルの更新を忘れる

```python
# NG: Q値だけ更新してモデルを更新していない
def update(self, state, action, reward, next_state, done):
    self._update_q(state, action, reward, next_state, done)
    # self.model.store(state, action, next_state, reward)  ← 忘れている!
    for _ in range(self.n_planning):
        sim_s, sim_a, sim_ns, sim_r = self.model.simulate()
        self._update_q(sim_s, sim_a, sim_r, sim_ns, False)
```

**対策**: モデル学習と計画は必ずセットで行う。Q値更新 → モデル更新 → 想像の順序を守る。

#### エラー2: 想像でゴール到達を正しく処理しない

```python
# NG: 想像の経験で常に done=False としてしまう
for _ in range(self.n_planning):
    sim_s, sim_a, sim_ns, sim_r = self.model.simulate()
    self._update_q(sim_s, sim_a, sim_r, sim_ns, done=False)  # ← ゴール到達時も False
```

**対策**: 想像の経験でも、次の状態がゴールの場合は `done=True` として Q 値を更新する。

#### エラー3: epsilon を固定したままにする

```python
# 改善の余地あり: 学習が進んでも同じ探索率
epsilon = 0.1  # 常に10%ランダム
```

**対策**: 学習の進行に伴い epsilon を徐々に減少させる（epsilon decay）。ただし、今回の実装では簡潔さのために固定値としている。

### 10.3 確認クイズ（5問）

---

**Q1**: モデルフリー RL とモデルベース RL の最大の違いは何ですか？

<details>
<summary>回答を表示</summary>

**モデルフリー RL** は環境のモデル（遷移関数・報酬関数）を学習せず、実際の経験だけから直接方策や価値関数を学習します。一方、**モデルベース RL** は環境のモデルを学習し、そのモデルを使って仮想的な経験（想像）を生成して追加の学習を行います。

</details>

---

**Q2**: 世界モデルの3つの要素を挙げてください。

<details>
<summary>回答を表示</summary>

1. **表現モデル（Representation Model）**: 観測を潜在状態に変換する
2. **遷移モデル（Transition Model）**: 現在の状態と行動から次の状態を予測する
3. **報酬モデル（Reward Model）**: 現在の状態と行動から報酬を予測する

</details>

---

**Q3**: Dyna-Q で imagination 回数 N=0 とした場合、アルゴリズムは何と等価になりますか？

<details>
<summary>回答を表示</summary>

**通常の Q-Learning** と等価になります。N=0 の場合、想像のステップが実行されないため、実経験のみから Q 値を更新するモデルフリーの Q-Learning と同じ動作をします。

</details>

---

**Q4**: imagination 回数を無制限に増やし続けると、常に学習が改善されますか？その理由も説明してください。

<details>
<summary>回答を表示</summary>

**いいえ**、無制限に増やしても常に改善されるわけではありません。理由：

1. **収穫逓減**: 同じ経験からの学習は飽和する
2. **モデル誤差**: 学習初期の不正確なモデルから想像すると、誤った方策を学習するリスクがある
3. **計算コスト**: N に比例して計算量が増加するが、学習の改善は比例しない

</details>

---

**Q5**: モデルベース RL において、モデル誤差が長いホライズンで蓄積する問題に対する対策を一つ挙げてください。

<details>
<summary>回答を表示</summary>

主な対策の例：
- **短いホライズンで計画する**: 長期予測を避け、短い未来だけを想像する
- **アンサンブルモデル**: 複数のモデルを学習し、不確実性を推定する
- **Dyna スタイルの1ステップ計画**: 1ステップの遷移だけを使い、長い rollout を避ける（今回の Dyna-Q がまさにこのアプローチ）

</details>

---

## 参考文献

1. Sutton, R. S. (1991). Dyna, an Integrated Architecture for Learning, Planning, and Reacting. *SIGART Bulletin*, 2(4), 160-163.
2. Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.
3. Ha, D., & Schmidhuber, J. (2018). World Models. *arXiv:1803.10122*.
4. Hafner, D., et al. (2023). Mastering Diverse Domains through World Models. *arXiv:2301.04104*.